Cell 1 — Check GPU and install dependencies

In [ ]:
import torch
from IPython.display import Markdown, display

display(Markdown("## System Check"))
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

#Necesario si se entrena en google collab
%pip install git+https://github.com/huggingface/transformers
%pip install git+https://github.com/huggingface/accelerate
%pip install unsloth unsloth_zoo
%pip install huggingface_hub datasets trl peft bitsandbytes


Cell 2 — Load model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

display(Markdown("## Loading Qwen2.5-7B-Instruct with Unsloth"))

#model_name = "Qwen/Qwen2.5-1.5B-Instruct"
model_name = "Qwen/Qwen2.5-7B-Instruct"


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 1024,
    load_in_4bit = True,
)

print("Model loaded successfully")

Cell 3 — Attach LoRA adapters

In [ ]:
display(Markdown("## Attaching LoRA Adapters"))

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    #max_seq_length = 2048,
    max_seq_length = 1024,

)

print("LoRA adapters attached")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## CAI — Paso 1: Cargar Constitución y Prompts Red Team

> Se arga la constitución y los prompts directamente desde `codigo_moral_v2_1.json`.
> Hay que subir el archivo a Colab antes de ejecutar esta celda.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, random
from IPython.display import Markdown, display

display(Markdown("## Cargando Constitución Cultural y Prompts Red Team"))

# ── Cargar desde el JSON del código moral ─────────────────────────────────────
#JSON_PATH = "/content/codigo_moral_v2_1.json"
JSON_PATH = "/content/drive/MyDrive/2026/Experimentos_2026/Alineacion_Cosmovisiones/Github_Adrian_CAI/Constitucion_Operativa/codigo_moral_v2.1.json"

with open(JSON_PATH, encoding="utf-8") as f:
    codigo_moral = json.load(f)

# Constitución: pares CritiqueRequest / RevisionRequest
CONSTITUTION = [
    {
        "CritiqueRequest": p["constitution_critique"],
        "RevisionRequest": p["constitution_revision"],
        "principio"      : p["principio"],
    }
    for p in codigo_moral
    if p.get("constitution_critique") and p.get("constitution_revision")
]

# Prompts red team: los casos concretos de cada situación
RED_TEAM_PROMPTS = [
    s["caso"]
    for p in codigo_moral
    for s in p.get("situaciones_concretas", [])
]

print(f"Constitución cargada   : {len(CONSTITUTION)} principios")
print(f"Prompts red team       : {len(RED_TEAM_PROMPTS)} prompts")
print()
print("Principios cargados:")
for i, c in enumerate(CONSTITUTION):
    print(f"  {i+1:2}. {c['principio']}")
print()
print("Ejemplo de principio:")
print(f"  Critique : {CONSTITUTION[0]['CritiqueRequest'][:80]}...")
print(f"  Revision : {CONSTITUTION[0]['RevisionRequest'][:80]}...")


## CAI — Paso 2: Loop Crítica → Revisión

> Usa los principios del código moral para autocriticar y revisar cada respuesta.


In [ ]:
import torch
from IPython.display import Markdown, display

display(Markdown("## Loop Constitucional: Crítica → Revisión"))

N_REVISIONS = 2

def generate_response(prompt_text, max_new_tokens=256):
    messages = [{"role": "user", "content": prompt_text}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True,
        add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs, max_new_tokens=max_new_tokens,
            temperature=1.0, do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs.shape[1]:], skip_special_tokens=True
    ).strip()


def constitutional_critique_revision(original_prompt, current_response, principle):
    # Crítica
    critique_prompt = (
        f"La siguiente es una conversación entre un usuario y un asistente:\n\n"
        f"Usuario: {original_prompt}\n"
        f"Asistente: {current_response}\n\n"
        f"Solicitud de crítica: {principle['CritiqueRequest']}\n"
        f"Crítica:"
    )
    critique = generate_response(critique_prompt, max_new_tokens=150)

    # Revisión
    revision_prompt = (
        f"La siguiente es una conversación entre un usuario y un asistente:\n\n"
        f"Usuario: {original_prompt}\n"
        f"Asistente: {current_response}\n\n"
        f"Crítica: {critique}\n\n"
        f"Solicitud de revisión: {principle['RevisionRequest']}\n"
        f"Respuesta revisada:"
    )
    revision = generate_response(revision_prompt, max_new_tokens=256)
    return critique, revision


def process_red_team_prompt(original_prompt, n_revisions=N_REVISIONS):
    initial_response = generate_response(original_prompt)
    current_response = initial_response
    all_critiques, all_revisions = [], []

    for _ in range(n_revisions):
        # Samplear principio aleatorio de la constitución cultural
        principle = random.choice(CONSTITUTION)
        critique, revision = constitutional_critique_revision(
            original_prompt, current_response, principle
        )
        all_critiques.append(critique)
        all_revisions.append(revision)
        current_response = revision

    return {
        "prompt"          : original_prompt,
        "initial_response": initial_response,
        "critiques"       : all_critiques,
        "revisions"       : all_revisions,
        "completion"      : all_revisions[-1],
    }


print(f"Procesando {len(RED_TEAM_PROMPTS)} prompts con {N_REVISIONS} revisiones c/u...")
constitutional_dataset = []

for idx, prompt in enumerate(RED_TEAM_PROMPTS):
    print(f"[{idx+1}/{len(RED_TEAM_PROMPTS)}] {prompt[:60]}...")
    result = process_red_team_prompt(prompt)
    constitutional_dataset.append(result)
    print(f"  Inicial  : {result['initial_response'][:80]}...")
    print(f"  Revisión : {result['completion'][:80]}...")
    print()

print(f"Loop constitucional completo: {len(constitutional_dataset)} ejemplos")


##  CAI — Paso 3: Combinar y Guardar Dataset

> Combina el dataset cultural base con los ejemplos generados por el loop CAI.
> El dataset final mezcla conocimiento cultural + respuestas revisadas.


In [ ]:
import json, os
from IPython.display import Markdown, display

display(Markdown("## Combinando y Guardando Dataset"))

FINAL_DATASET = "/content/dataset_sft_cultural.jsonl"

# ── Cargar dataset cultural base (situaciones_concretas del JSON) ─────────────
dataset_cultural = []
for p in codigo_moral:
    for s in p.get("situaciones_concretas", []):
        dataset_cultural.append({
            "prompt"    : s["caso"],
            "completion": s["respuesta_alineada"],
        })

# ── Agregar ejemplos del loop CAI ─────────────────────────────────────────────
dataset_cai = [
    {"prompt": item["prompt"], "completion": item["completion"]}
    for item in constitutional_dataset
]

# ── Combinar y guardar ────────────────────────────────────────────────────────
dataset_final = dataset_cultural + dataset_cai

with open(FINAL_DATASET, "w", encoding="utf-8") as f:
    for item in dataset_final:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

# Guardar dataset completo con críticas para análisis
with open("/content/constitutional_dataset_full.jsonl", "w", encoding="utf-8") as f:
    for item in constitutional_dataset:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Dataset cultural base   : {len(dataset_cultural)} ejemplos")
print(f"Dataset loop CAI        : {len(dataset_cai)} ejemplos")
print(f"Dataset final combinado : {len(dataset_final)} ejemplos")
print(f"   Guardado en: {FINAL_DATASET}")

# Actualizar file_name para Cell 4
file_name = "dataset_sft_cultural.jsonl"
print(f"\n file_name → '{file_name}'")


# Guardando la salida en drive

In [ ]:
import shutil
shutil.copy('/content/dataset_sft_cultural.jsonl',
            '/content/drive/MyDrive/2026/Experimentos_2026/Alineacion_Cosmovisiones/salidas_entrenamiento/dataset_sft_cultural.jsonl')
print("Dataset guardado en Drive")

In [ ]:
from datasets import load_dataset
from IPython.display import Markdown, display

display(Markdown("## Loading Dataset"))

# dataset_sft_cultural.jsonl es generado por la Cell CAI-C
file_name = "dataset_sft_cultural.jsonl"
print(f"Cargando: {file_name}")

dataset = load_dataset(
    "json",
    data_files={"train": file_name},
    split="train"
)

print(f"✅ Dataset loaded: {len(dataset)} records")
print("\n--- Sample record ---")
print(f"Prompt: {dataset[0]['prompt']}")
print(f"Completion: {dataset[0]['completion']}")


Cell 5 — Format dataset

In [ ]:
def format_example(example):
    messages = [
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["completion"]},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False
        )
    }
dataset = dataset.map(format_example)

dataset = dataset.remove_columns(
    [col for col in dataset.column_names if col != "text"]
)

Cell 6 - Set up variables and record training parameters

In [ ]:
import pandas as pd
import numpy as np
import librosa.display, os
import matplotlib.pyplot as plt
%matplotlib inline
import hashlib


try:
    df_log = pd.read_csv("log.csv")
except FileNotFoundError:
    df_log = pd.DataFrame(columns=[
        "ID_Exp",
        "Hash",
        "Model",
        "entry_file",
        "max_seq_length",
        "per_device_train_batch_size",
        "gradient_accumulation_steps",
        "warmup_steps",
        "num_train_epochs",
        "logging_steps",
        "seed",
        "dataset_num_proc"
        ])

df_log.to_csv("log.csv", index=False)

max_seq_length = 1024
per_device_train_batch_size = 1
gradient_accumulation_steps = 4
warmup_steps = 5
num_train_epochs = 6
logging_steps = 10
seed = 1500
dataset_num_proc = 1

hash = hashlib.md5(f"{model_name},{max_seq_length},{per_device_train_batch_size},{gradient_accumulation_steps}.{warmup_steps},{num_train_epochs},{logging_steps},{seed},{dataset_num_proc}".encode()).hexdigest()
hash

Cell 7 — Train

In [ ]:
from trl import SFTTrainer, SFTConfig

display(Markdown("## Starting Fine-tuning"))


trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    tokenizer = tokenizer,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = per_device_train_batch_size,
        gradient_accumulation_steps = gradient_accumulation_steps,
        warmup_steps = warmup_steps,
        num_train_epochs = num_train_epochs,
        logging_steps = logging_steps,
        output_dir = "/content/output",
        optim = "adamw_8bit",
        seed = seed,
        dataset_num_proc = dataset_num_proc,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        packing = True,
    ),
)

trainer_stats = trainer.train()
print(f"Training complete!")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.0f} seconds")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")


Cell 8 - Save experiment info

In [ ]:
import pandas as pd
df_log = pd.read_csv("log.csv")
experiment = f"EXP{len(df_log)}"
new_exp = {
    "ID_Exp": experiment,
    "Hash": hash,
    "Model": model_name,
    "entry_file": file_name,
    "max_seq_length": max_seq_length,
    "per_device_train_batch_size": per_device_train_batch_size,
    "gradient_accumulation_steps": gradient_accumulation_steps,
    "warmup_steps": warmup_steps,
    "num_train_epochs": num_train_epochs,
    "logging_steps": logging_steps,
    "seed": seed,
    "dataset_num_proc": dataset_num_proc
}

df_log = pd.concat([df_log, pd.DataFrame([new_exp])], ignore_index=True)
df_log.to_csv("log.csv",index=False)

*Cell* 7 — Test model before saving (optional)

In [ ]:
import pandas as pd
import os

display(Markdown("## Testing: Base Model vs Fine-tuned Model"))

FastLanguageModel.for_inference(model)

def ask(question):
    messages = [
        {
            "role": "system",
            "content": "Da la respuesta en de 3 a 5 oraciones."
        },
        {
            "role": "user",
            "content": question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

    response = tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens=True
    )
    return response


with open("/content/Preguntas.txt", "r", encoding="utf-8") as f:
    questions = [line.strip() for line in f if line.strip()]

answers = [ask(q) for q in questions]

columns = ["experiment","hash"] + questions

row = [experiment, hash] + answers


output_path = "/content/respuestas.csv"

df = pd.DataFrame([row], columns=columns)
if os.path.exists(output_path):
    df_existing = pd.read_csv(output_path)
    df_final = pd.concat([df_existing, df], ignore_index=True)
else:
    df_final = df

df_final.to_csv(output_path, index=False)
print(f"Excel actualizado: {output_path}")



Cell 8 — Save the model

In [ ]:
display(Markdown("## Saving Fine-tuned Model"))

# Save LoRA adapters only
model.save_pretrained("/content/modelo_lora")
tokenizer.save_pretrained("/content/modelo_lora")



print("Model saved!")
print("LoRA adapters: /content/modelo_lora")

In [ ]:
!zip -r modelo_lora.zip /content/modelo_lora



# Guarda en drive

In [ ]:
shutil.copy('/content/modelo_lora.zip',
            '/content/drive/MyDrive/2026/Experimentos_2026/Alineacion_Cosmovisiones/salidas_entrenamiento/modelo_lora.zip')

print("✅ modelo_lora.zip guardado en Drive")